### 1. Introducción a la Modulación OFDM

La **Modulación por División de Frecuencia Ortogonal (OFDM)** es una técnica de modulación digital multiportadora que divide un canal de banda ancha en un gran número de subcanales de banda estrecha, conocidos como **subportadoras**. La clave de OFDM reside en la **ortogonalidad** de estas subportadoras, lo que permite que sus espectros se solapen sin causar interferencia entre sí, logrando una alta eficiencia espectral.

Las principales ventajas que han convertido a OFDM en la base de estándares modernos como Wi-Fi (802.11a/g/n/ac/ax), LTE, 5G y DVB son:

*   **Robustez contra la Interferencia Intersimbólica (ISI):** Al transmitir datos en paralelo sobre muchas subportadoras lentas, la duración de cada símbolo se incrementa significativamente. Esto hace que el retardo del canal (multipath delay spread) sea una fracción pequeña de la duración del símbolo, mitigando la ISI.
*   **Eliminación de la ISI con el Prefijo Cíclico (CP):** Se introduce un intervalo de guarda (Guard Interval) antes de cada símbolo, usualmente una copia del final del propio símbolo (Prefijo Cíclico). Esto cumple dos funciones:
    1.  Absorbe la ISI del símbolo anterior, dejando la parte útil del símbolo actual intacta.
    2.  Convierte la convolución lineal del canal en una convolución circular, lo que simplifica enormemente la ecualización en el receptor.
*   **Eficiencia Computacional:** En lugar de utilizar un banco de moduladores y demoduladores analógicos (uno por subportadora), OFDM se implementa eficientemente en el dominio digital mediante la **Transformada Inversa Rápida de Fourier (IFFT)** en el transmisor y la **Transformada Rápida de Fourier (FFT)** en el receptor.
*   **Ecualización Sencilla:** Gracias al prefijo cíclico, el efecto de un canal selectivo en frecuencia se reduce a una simple multiplicación por un escalar complejo en cada subportadora. Esto permite una ecualización de un solo toque (one-tap equalizer) por subportadora, mucho más simple que los ecualizadores complejos necesarios en sistemas de portadora única.

### 2. Flujo de Transmisión y Recepción (Visión Ideal)

El proceso completo, desde los bits de entrada hasta los bits recuperados, sigue una serie de pasos bien definidos.

#### **Proceso del Transmisor OFDM:**

1.  **Generación de Bits:** Se crea una secuencia de bits binarios aleatorios como fuente de datos.
2.  **Mapeo de Símbolos:** Los bits se agrupan y se mapean a símbolos complejos de una constelación digital (ej. QPSK, 16-QAM). Para QPSK, se agrupan de 2 en 2.
3.  **Conversión Serie a Paralelo (S/P):** Los símbolos complejos se agrupan en bloques de tamaño `N`, donde `N` es el número de subportadoras. Cada bloque formará un símbolo OFDM.
4.  **Modulación IFFT:** Se aplica la IFFT a cada bloque de `N` símbolos. Esto transforma los datos del dominio de la frecuencia al dominio del tiempo, generando la señal multiportadora. La salida son `N` muestras de tiempo complejas.
5.  **Inserción del Prefijo Cíclico (CP):** Se copian las últimas `L` muestras de la salida de la IFFT y se anteponen al bloque. El símbolo OFDM ahora tiene `N + L` muestras.
6.  **Conversión Paralelo a Serie (P/S):** Los símbolos OFDM se concatenan para formar la trama de datos a transmitir.

#### **Proceso del Receptor OFDM:**

1.  **Sincronización y Conversión S/P:** El receptor se sincroniza con la trama recibida y la divide en bloques de `N + L` muestras.
2.  **Eliminación del Prefijo Cíclico:** Se descartan las primeras `L` muestras de cada bloque, eliminando el CP.
3.  **Demodulación FFT:** Se aplica la FFT a los `N` muestras restantes para transformar la señal de vuelta al dominio de la frecuencia. La salida son los `N` símbolos complejos recibidos en cada subportadora.
4.  **Ecualización (Opcional):** Se corrige la atenuación y el desfase introducidos por el canal (en un canal ideal, este paso es una identidad).
5.  **Demapeo de Símbolos:** Cada símbolo complejo recibido se decide al punto más cercano de la constelación original (detección de máxima verosimilitud).
6.  **Conversión Paralelo a Serie y Recuperación de Bits:** Los símbolos decididos se convierten de nuevo a bits, y estos se concatenan para formar la secuencia de datos recuperada.
7.  **Cálculo de BER:** Se comparan los bits transmitidos con los recuperados para medir el rendimiento.

Con estos fundamentos establecidos, procederemos a implementar cada bloque en Python.

### 0. Parámetros globales

Sea  
\[
\begin{aligned}
N      &= \text{nº de subportadoras (tamaño FFT)}            \\
M      &= \text{orden de modulación (QPSK → 4)}  \\
\mu    &= \log_2 M \quad (\text{bits por símbolo})           \\
L_{cp} &= \text{longitud del prefijo cíclico}                \\
F_s    &= \text{frecuencia de muestreo (normalizada = 1)}    \\
T_u    &= \frac{N}{F_s}\quad (\text{duración útil})          \\
\Delta f &= \frac{1}{T_u}\quad (\text{espaciado entre subportadoras}) \\
T_{sym} &= T_u + \frac{L_{cp}}{F_s}\quad (\text{duración total símbolo})\\
\end{aligned}
\]


In [4]:
# 0_parámetros.py  (si prefieres un script) ────────────────────────────
import numpy as np

RNG = np.random.default_rng(seed=42)   # reproducibilidad

# --- Parámetros base ---------------------------------------------------
N         = 64        # tamaño FFT
M         = 4         # QPSK
mu        = int(np.log2(M))
L_cp      = N // 4    # prefijo cíclico (25 %)
N_sym     = 10_000    # símbolos OFDM a simular
Fs        = 1.0       # frecuencia de muestreo normalizada
Es        = 1.0       # energía media por símbolo después del mapeo

# --- Derivados ---------------------------------------------------------
Tu        = N / Fs              # duración útil
T_cp      = L_cp / Fs           # duración del CP
T_sym     = Tu + T_cp           # duración total
delta_f   = 1.0 / Tu            # espaciado de subportadoras

# Diccionario de configuración
# agrupa las variables en un unico objeto
PARAMS = {
    "N": N, "M": M, "mu": mu, "L_cp": L_cp, "N_sym": N_sym, "Fs": Fs,
    "Es": Es, "Tu": Tu, "T_cp": T_cp, "T_sym": T_sym, "delta_f": delta_f,
}

# --- Sanity checks -----------------------------------------------------
assert 2 ** mu == M, "M no es potencia de 2"
assert L_cp < N,      "El CP no puede ser ≥ N"
print("Parámetros OK")
# ----------------------------------------------------------------------


Parámetros OK
